<a href="https://colab.research.google.com/github/nurfnick/NetworkScience/blob/main/HomeworkAssignments/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CS 5483 Network Science — Project 3
### Algorithms, Measurement Error, Real‑World Structure, and Random Graph Models (Instructions‑Only)

**Allowed libraries:** `networkx`, `numpy`, `matplotlib`. Additional packages require citation and justification in the write‑up.
**Students must write all code.**



## Dataset(s)
- Choose one **real network** (5k–50k edges recommended). Cite the source.
- Generate synthetic graphs in later parts for comparison.

**Show here:**
`|V|`, `|E|`, density, component summary, and any preprocessing choices.


In [ ]:

# TODO: Imports (NetworkX, NumPy, Matplotlib)
# import networkx as nx
# import numpy as np
# import matplotlib.pyplot as plt
# import random, time

# TODO: Load REAL network into G
# G = ...

# TODO: Print stats and components summary
# print(...)


In [1]:
import networkx as nx
import pandas as pd
import random

I have choosen the SNAP Social Circles Facebook dataset and unpacked the facebook_combined data into my github where I load below and create out graph that we will use throughout the project.

In [2]:



url = "https://raw.githubusercontent.com/nurfnick/NetworkScience/refs/heads/main/HomeworkAssignments/facebook_combined.txt"



G = nx.Graph()

df = pd.read_csv(url, header=None, sep=' ')
df.columns = ["node1", "node2"]


G.add_edges_from(df.values.tolist())

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Density of Graph edges: { nx.density(G)}")
print(f"Number of components: {nx.number_connected_components(G)}")

Number of nodes: 4039
Number of edges: 88234
Density of Graph edges: 0.010819963503439287
Number of components: 1


I did no preprocessing to the data beyond unpacking the data.


## Part 1 — Graph Algorithms & Complexity (25%)
Implement:
1) Representations (edge list / adjacency list / adjacency matrix) with brief tradeoff discussion.
2) DFS/BFS: components; **triangle counting**; **diameter approximation** (explain method).
3) Shortest paths: **Dijkstra** from 5 sources; **Floyd–Warshall** APSP only if `|V| ≤ 1200`, else justify alternative.
4) **Betweenness centrality** (full or sampled, with sample size stated).
5) **Runtime study**: record times vs size (down‑sampled subgraphs) and relate to big‑O.


In [ ]:

# TODO: Implement components, triangle count, diameter approximation
# TODO: Implement Dijkstra (5 sources) and APSP strategy
# TODO: Compute betweenness (top‑5)
# TODO: Collect and present runtimes (table/figure) and commentary


I'll create the differing represnetations for edges.  
1. The edge list which was actually used to create the graph `G` is already here and loaded in a dataframe.  It makes it easy to find neighbors and also great for changing the structure of a network like adding or deleting edges.
2. Adjacency list is a linked list.  Again it is efficient for finding neighbors and graph traversal.
3. Adjacency matrix is the last structure and it is great for looking up neighbors but has large storage especially in networks with sparse edges.  Hard to delete edges for sure!

In [3]:
count = 0
for i in G.adjacency():
  print(i)
  count += 1
  if count > 5:
    break


(0, {1: {}, 2: {}, 3: {}, 4: {}, 5: {}, 6: {}, 7: {}, 8: {}, 9: {}, 10: {}, 11: {}, 12: {}, 13: {}, 14: {}, 15: {}, 16: {}, 17: {}, 18: {}, 19: {}, 20: {}, 21: {}, 22: {}, 23: {}, 24: {}, 25: {}, 26: {}, 27: {}, 28: {}, 29: {}, 30: {}, 31: {}, 32: {}, 33: {}, 34: {}, 35: {}, 36: {}, 37: {}, 38: {}, 39: {}, 40: {}, 41: {}, 42: {}, 43: {}, 44: {}, 45: {}, 46: {}, 47: {}, 48: {}, 49: {}, 50: {}, 51: {}, 52: {}, 53: {}, 54: {}, 55: {}, 56: {}, 57: {}, 58: {}, 59: {}, 60: {}, 61: {}, 62: {}, 63: {}, 64: {}, 65: {}, 66: {}, 67: {}, 68: {}, 69: {}, 70: {}, 71: {}, 72: {}, 73: {}, 74: {}, 75: {}, 76: {}, 77: {}, 78: {}, 79: {}, 80: {}, 81: {}, 82: {}, 83: {}, 84: {}, 85: {}, 86: {}, 87: {}, 88: {}, 89: {}, 90: {}, 91: {}, 92: {}, 93: {}, 94: {}, 95: {}, 96: {}, 97: {}, 98: {}, 99: {}, 100: {}, 101: {}, 102: {}, 103: {}, 104: {}, 105: {}, 106: {}, 107: {}, 108: {}, 109: {}, 110: {}, 111: {}, 112: {}, 113: {}, 114: {}, 115: {}, 116: {}, 117: {}, 118: {}, 119: {}, 120: {}, 121: {}, 122: {}, 123: 

In [4]:
nx.adjacency_matrix(G).todense()

array([[0, 1, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

Now onto components.  We have only one component here.  We have already seen that only one component exists.  We utilize the built ins to compute the number of triangles.  Interestingly we see that 0 has lots of triangles but 11 has none!

In [5]:
list(nx.triangles(G).items())[0:12]

[(0, 2519),
 (1, 57),
 (2, 40),
 (3, 86),
 (4, 39),
 (5, 26),
 (6, 14),
 (7, 82),
 (8, 19),
 (9, 634),
 (10, 37),
 (11, 0)]

For diameter, I do the following with a builtin.  This was LONG computation time!!!  I think I'll explore this and try to do a different algorithm.

In [ ]:
nx.diameter(G)

8

So I am going to randomly pick some nodes and find the node they are furthest from.  I'll update my diameter everytime I find one that is the furthest.

In [6]:
dist = 0
numedges = 10
l = random.choices(list(G.nodes()), k=numedges)

for edge in l:
  for node in list(G.nodes()):
    p = nx.shortest_path_length(G, source=edge, target=node)
    if p > dist:
      dist = p

dist

7

Um, I found the longest in several iterations of this!  That's fairly random...  I find this algorithm clunky.  I could do much better if I built the tree for the 10 nodes I picked and then look at the length of it.  I think this algorithm is building that tree for each of the nodes.  This is a clunky way to do it but kinda works.  This algorithm is $O(n^2)*O(shortestpath)$.  I am going to improve on this by creating a breadth first algorithm.

In [7]:
def bfs(node):
  dist = [0]*(G.number_of_nodes())
  q = [node]
  visited = [False]*(G.number_of_nodes())
  visited[node] = True
  while q:
    u = q.pop(0)
    for v in G.neighbors(u):
      if not visited[v]:
        visited[v] = True
        dist[v] = dist[u] + 1
        q.append(v)
  return max(dist)

bfs(1010)

6

Now I use this to repeat teh experiment for the diameter.

In [8]:
dist = 0
numedges = 10
l = random.choices(list(G.nodes()), k=numedges)

for edge in l:
  p = bfs(edge)
  if p > dist:
    dist = p

dist

7

I do get a bit of variety when repeating this experiment but it is generally close to 8.  The big O for bredth fist is O(V + E).  For diameter then it will be O(V(V+E)).  We do that here below

In [ ]:
for edge in G.nodes():
  p = bfs(edge)
  if p > dist:
    dist = p

dist

8

If I wanted to find the shortest path, I do the dijkstra's algorithm.  I'll start with a node and add the distance to every neighbor as 1 (since there are no weights here) Then I'll go to the next node and find all it's neighbors but only update distances if the new distance is less.  Then we repeat the algorithm until we have gone to all the nodes.  To deal with distances between nodes not being one, there is only a small tweak required in the code below.

In [11]:
def dijkstra(node):
  dist = [float('inf')]*(G.number_of_nodes())
  dist[node] = 0
  q = [node]
  visited = [False]*(G.number_of_nodes())
  visited[node] = True
  while q:
    u = q.pop(0)
    for v in G.neighbors(u):
      if not visited[v]:
        visited[v] = True
        if dist[u] + 1 < dist[v]:
          dist[v] = dist[u] + 1
        q.append(v)
  return dist

max(dijkstra(0))

6

Next we look a Floyd-Wallace algorithm.  We simply iterate through all nodes and keep track of parent for each of them.

In [28]:
def floyd_warshall(G):
  dist = [[float('inf') for _ in range(G.number_of_nodes())] for _ in range(G.number_of_nodes())]
  parent = [[float('inf') for _ in range(G.number_of_nodes())] for _ in range(G.number_of_nodes())]
  for i in range(G.number_of_nodes()):
    dist[i][i] = 0
    parent[i][i] = i
  for u in G.nodes():
    for v in G.neighbors(u):
      dist[u][v] = 1
      parent[u][v] = u
  for k in range(G.number_of_nodes()):
    for i in range(G.number_of_nodes()):
      for j in range(G.number_of_nodes()):
        if dist[i][j] > dist[i][k] + dist[k][j]:
          dist[i][j] = dist[i][k] + dist[k][j]
          parent[i][j] = k
  return dist, parent

dist, parent = floyd_warshall(G.subgraph(range(21)))

In [29]:
dist

[[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1],
 [1, 2, 2, 0, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2],
 [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 1],
 [1, 2, 2,

In [30]:
parent

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2],
 [3, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [6, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [7, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [9, 0, 0, 9, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 0, 0, 0, 0, 0, 0, 0, 0],
 [13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 0, 0, 0, 0, 0, 0, 0],
 [14, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14, 0, 0, 0, 0, 0, 14],

This is working well, I didn't get results on my whole dataset so I have used a subgraph here for testing.  We can see that 2 and 20 are neighbors!

In [32]:
def betweenness_centrality(graph):
    """
    Computes the betweenness centrality for all nodes in the graph.

    Args:
        graph: A NetworkX graph.

    Returns:
        A dictionary where keys are nodes and values are their betweenness centrality.
    """
    betweenness = {node: 0.0 for node in graph.nodes()}
    nodes = list(graph.nodes())

    for s in nodes:
        # Stack for DFS
        stack = []
        # List of predecessors
        pred = {v: [] for v in nodes}
        # Distance from source
        dist = {v: -1 for v in nodes}
        # Number of shortest paths from source
        sigma = {v: 0 for v in nodes}

        dist[s] = 0
        sigma[s] = 1
        queue = [s]

        while queue:
            v = queue.pop(0)
            stack.append(v)

            for w in graph.neighbors(v):
                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    queue.append(w)
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    pred[w].append(v)

        delta = {v: 0.0 for v in nodes}
        while stack:
            w = stack.pop()
            for v in pred[w]:
                delta[v] += (sigma[v] / sigma[w]) * (1 + delta[w])
            if w != s:
                betweenness[w] += delta[w]

    # Normalize betweenness centrality
    n = graph.number_of_nodes()
    scale = (n - 1) * (n - 2) / 2.0
    for v in betweenness:
        betweenness[v] /= scale

    return betweenness

# Example usage (consider sampling for larger graphs)
betweenness_scores = betweenness_centrality(G.subgraph(range(100))) # Example with subgraph
print(betweenness_scores)

{0: 1.8103189327679128, 1: 0.0008726723012437296, 2: 0.0, 3: 0.0005153576582148011, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0017865732151446435, 8: 0.0, 9: 0.005521689195158583, 10: 0.0, 11: 0.0, 12: 0.0, 13: 0.0011750154607297464, 14: 0.0001374287088572803, 15: 0.0, 16: 0.0, 17: 0.0, 18: 0.0, 19: 0.00041228612657184083, 20: 0.0007558578987150415, 21: 0.004838472185410961, 22: 0.0, 23: 0.0010307153164296021, 24: 0.0002748574177145606, 25: 0.02747985196964789, 26: 0.005447085038921775, 27: 0.0, 28: 0.0, 29: 0.00030921459492888067, 30: 0.0005840720126434411, 31: 0.001958359101216244, 32: 0.0, 33: 0.0, 34: 0.0, 35: 0.0, 36: 0.0, 37: 0.0, 38: 0.0, 39: 6.871435442864014e-05, 40: 0.0026013291319413765, 41: 0.0033670033670033673, 42: 0.0, 43: 0.0, 44: 0.0, 45: 0.0, 46: 0.0, 47: 0.0, 48: 0.0027142169999312855, 49: 0.0, 50: 0.0, 51: 0.00041228612657184083, 52: 0.0, 53: 0.0015941730227444512, 54: 0.0006184291898577613, 55: 0.0, 56: 0.018131754866448743, 57: 0.00018552875695732838, 58: 0.0, 59: 0.0, 60: 0.0

Okay, let's analyze the runtime and Big O complexity of the algorithms you've implemented so far.

1. Adjacency List (Implicit in NetworkX G.adjacency()):

    Runtime/Big O: Iterating through G.adjacency() is essentially iterating through the adjacency list representation. The time complexity to iterate through all neighbors of a node u is O(degree(u)). To iterate through all nodes and a few neighbors, as in your example, the complexity is proportional to the number of nodes and edges visited, which is roughly O(|V| + E_subset).

2. Adjacency Matrix (nx.adjacency_matrix(G).todense()):

    Runtime/Big O: Generating the adjacency matrix for a graph with |V| nodes takes O(|V|^2) time and space, regardless of the number of edges, as it creates a |V| x |V| matrix. Converting it to a dense matrix explicitly takes O(|V|^2).

3. Triangle Counting (nx.triangles(G)):

    Runtime/Big O: The NetworkX nx.triangles() function for an undirected graph has a time complexity of O(|V| + |E|^(3/2)). In dense graphs, this can be closer to O(|V|^3). Your example of showing the first 12 items is fast, but the computation of all triangles for the whole graph would take the full complexity.

4. Diameter Approximation (using nx.shortest_path_length):

    Runtime/Big O: Your first approach iterates through a sample of numedges nodes and for each, calculates the shortest path to all other nodes. The nx.shortest_path_length function in NetworkX for an unweighted graph uses BFS, which is O(|V| + |E|). Therefore, your loop has a complexity of numedges * O(|V| * (|V| + |E|)). If numedges is a constant, this is O(|V|(|V|+|E|)).

5. Your bfs function:

    Runtime/Big O: Your custom bfs function correctly implements Breadth-First Search. The time complexity for BFS on a graph is O(|V| + |E|). This is efficient for finding shortest paths in unweighted graphs.

6. Diameter Approximation (using your bfs function):

    Runtime/Big O: Your second approach using your bfs function iterates through a sample of numedges nodes and runs BFS from each. This results in a complexity of numedges * O(|V| + |E|). If numedges is a constant, this is O(|V| + |E|), which is a significant improvement over the previous approach.

7. Diameter Calculation (using your bfs function for all nodes):

    Runtime/Big O: When you run your bfs function for all nodes in the graph to find the true diameter, the complexity becomes O(|V| * (|V| + |E|)). This is because you are performing a BFS from each of the |V| nodes.

8. Your dijkstra function:

    Runtime/Big O: Your dijkstra function, as implemented, is essentially a BFS because you are assuming edge weights of 1 and using a simple queue. Therefore, its time complexity for an unweighted graph is O(|V| + |E|). A standard Dijkstra implementation with a priority queue on a graph with non-negative edge weights would have a complexity closer to O(|E| log |V|) or O(|E| + |V| log |V|) depending on the priority queue implementation.

9. Your floyd_warshall function:

    Runtime/Big O: Your implementation of the Floyd-Warshall algorithm has three nested loops iterating up to the number of nodes. This gives it a time complexity of O(|V|^3). This algorithm is suitable for smaller graphs or dense graphs where |E| is close to |V|^2, but it becomes very slow for large, sparse graphs like the one you are using (|V| = 4039). This is why you saw a long computation time and used a subgraph for testing.

10. Your betweenness_centrality function:

    Runtime/Big O: Your implementation of betweenness centrality appears to be based on Brandes' algorithm, which is efficient for sparse graphs. For an unweighted graph, Brandes' algorithm has a time complexity of O(|V||E|). For a dense graph, it's O(|V|^3). Given your graph is sparse, the O(|V||E|) complexity is likely the dominant factor. Running this on the full graph will be computationally expensive, which is why sampling or using a subgraph, as you noted, is a good strategy.

Overall, you have a good understanding of implementing these algorithms. The runtime observations you made, particularly for the diameter calculation using repeated BFS and the Floyd-Warshall algorithm on the full graph, align with their theoretical Big O complexities. For larger graphs, sticking to algorithms with complexities closer to O(|V| + |E|) or O(|V| log |V|) will be more practical.



## Part 2 — Connectivity & Flow (10%)
- 3 source–sink pairs: **max‑flow** and **min‑cut**.
- 5 random pairs: **k** vertex‑ or edge‑disjoint paths for k∈{2,3}.

**Output:** table of results + 1–2 sentence robustness interpretation.


In [ ]:

# TODO: Max‑flow / min‑cut for 3 pairs
# TODO: k‑disjoint paths checks for 5 pairs (k=2,3)
# TODO: Summarize in a table



## Part 3 — Measurement Error & Link Prediction (20%)
- Remove **10%** of edges (seeded) to form `G_obs`; optionally add ≤2% spurious edges.
- Implement at least three methods: Jaccard, Adamic–Adar or Preferential Attachment, **Katz (truncated)** or **Resource Allocation**.
- Evaluate with a balanced test set (removed edges vs sampled non‑edges). Report **AUC** and **Precision@k** (`k∈{50,100}`) and include one evaluation plot.


In [ ]:

# TODO: Build G_obs with controlled edge removals/additions
# TODO: Implement similarity scores; assemble candidate pairs
# TODO: Create labeled test set; compute AUC and Precision@k
# TODO: Plot one ROC‑style or Precision@k figure



## Part 4 — Real‑World Structure Analysis (25%)
- **Degree distributions** (log–log); estimate power‑law exponent via MLE or compare to lognormal; state method.
- **Centrality distributions**: degree, betweenness, closeness, eigenvector (figures + discussion).
- **Clustering**: global and distribution of local clustering.
- **Assortativity**: degree assortativity coefficient and interpretation.


In [ ]:

# TODO: Degree distribution plots + parameter estimation or model comparison
# TODO: Centrality distribution plots + brief discussion
# TODO: Clustering metrics and local clustering distribution
# TODO: Degree assortativity value and interpretation



## Part 5 — Random Graph Models & Comparison (15%)
Generate matched models and compare to the real graph:
- **ER/gnp** with `p ≈ 2m / (n(n−1))`
- **Preferential Attachment (BA)** with parameters approximating |E|
- **Configuration Model** using the real graph’s degree sequence (project to simple graph; note any changes)

Report for each: `|V|`, `|E|`, avg degree, **avg path length** (LCC), **clustering**, degree distribution (overlay OK). Provide a comparison table and short commentary.


In [ ]:

# TODO: Generate ER, BA, and Configuration graphs
# TODO: Compute requested statistics and plots
# TODO: Comparison table + commentary



## Part 6 — Mini Write‑up (2–3 pages) (5%)
Synthesize results across parts; cite datasets and any extra packages; note limitations (scalability, sampling, model assumptions).



## Rubric (100 pts)
- Part 1 Algorithms & Complexity — **25**
- Part 2 Connectivity & Flow — **10**
- Part 3 Measurement Error & Link Prediction — **20**
- Part 4 Real‑World Structure — **25**
- Part 5 Random Models Comparison — **15**
- Part 6 Write‑up — **5**

**Integrity & Reproducibility**
- Student‑written code required; prohibited libraries → 0 for affected parts.
- Notebook must run end‑to‑end; missing citations → −2 to −3.
- Include **LLM Usage Log** below.


## LLM Usage Log

1.  can you build an algorithm that will compute the betweenness centrality of my graph G

I was able to accept this code with limited modifications.  Mostly just changing comments and deleting superfolous coding.

In [ ]:
# TODO: Paste 2–3 prompts and notes on what was accepted vs modified.
